# SemHAR-Gate — Block A Research Notebook
## Full UCF101 + HMDB51: Audit → Temporal Sampling → Normalization → ResNet18 Cache → BiGRU → Temporal Attention

**Purpose.** This notebook is the permanent foundation for the later SemHAR-Gate blocks. It deliberately stops before semantic prompting, semantic projection, calibration, and adaptive gating. Later blocks should consume the artifacts produced here rather than rebuilding the dataset pipeline.

### What the student learns while running this notebook
1. Standard UCF101/HMDB51 split protocols.
2. Dataset integrity and leakage auditing.
3. Temporal segment sampling for video.
4. Spatial preprocessing and ImageNet normalization.
5. Frozen ResNet18 frame-level representation extraction.
6. Disk-efficient reusable feature caching.
7. BiGRU temporal modeling.
8. Temporal attention pooling.
9. Research-grade metrics, checkpointing, and export of video embeddings for future blocks.

### Research protocol
- **UCF101:** full 101 classes; official fold/split is used for train/test. Validation is created **only from the official training partition** using class-wise group separation based on UCF group IDs.
- **HMDB51:** full 51 classes; official fold/split is used for train/test. Validation is created **only from the official training partition** using a fixed stratified split.
- Test data are never used for preprocessing statistics, model selection, or hyperparameter tuning.
- Default fold is **1**. The same notebook supports folds 2 and 3 later.

> **Important:** Full UCF101 + HMDB51 require substantial disk space and preprocessing time. `QUICK_MODE=False` is the research setting. Use quick mode only to verify the pipeline.

## 1. Environment setup

This cell installs lightweight Python dependencies. It intentionally **does not reinstall PyTorch/Torchvision**, because GPU-enabled PyTorch installations differ by CUDA version. Colab/Kaggle usually already provide compatible versions.

If `torch` or `torchvision` is missing, install the correct build from the official PyTorch installation page before continuing.

In [1]:
# Lightweight dependencies only.
# Re-running this cell is safe.
%pip install -q pandas scikit-learn opencv-python-headless pillow tqdm huggingface_hub pyarrow matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports, reproducibility, and hardware check

A research pipeline must make randomness explicit. We seed Python, NumPy, and PyTorch. Deterministic kernels can reduce speed, so this notebook uses deterministic random seeds without forcing every CUDA operation into deterministic mode.

In [2]:
import os
import sys
import math
import json
import time
import random
import hashlib
import shutil
import zipfile
import subprocess
import platform
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode

warnings.filterwarnings("ignore")

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

d:\Projects\ML_project\action-recognition-nlp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.12.10
Platform: Windows-10-10.0.19045-SP0
PyTorch: 2.13.0+cpu
Torchvision: 0.28.0+cpu
CUDA available: False


In [3]:
def seed_everything(seed: int = 42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

SEED = 42
seed_everything(SEED)

## 3. Central configuration — keep this for future blocks

Later semantic and gating blocks should import/reuse the same workspace, manifests, class mappings, cached frame features, and trained visual encoder outputs.

### Before the first full run
- Leave `QUICK_MODE=False` for research.
- Set `AUTO_DOWNLOAD=True` if the datasets are not already available.
- If you already have extracted datasets, set `AUTO_DOWNLOAD=False` and edit the paths in `DATASET_PATHS`.
- `FOLD=1` is the first benchmark fold. Later conference experiments can repeat with folds 2 and 3.

The Hugging Face download helper below uses mirrors of the original archives for convenience. The notebook still preserves the official split files.

In [4]:
@dataclass
class Config:
    seed: int = 42
    fold: int = 1

    # Full research mode by default.
    quick_mode: bool = False
    quick_train_per_class: int = 8
    quick_eval_per_class: int = 4

    # Dataset acquisition.
    auto_download: bool = False   # Set True if archives are not already present.
    datasets_to_prepare: tuple = ("ucf101", "hmdb51")
    datasets_to_train: tuple = ("ucf101", "hmdb51")

    # Validation is carved only from official TRAIN.
    val_fraction: float = 0.10

    # Audit.
    audit_workers: int = 1   # 1 is safest across OSes/codecs; increase cautiously.
    audit_quick_hash: bool = True
    audit_decode_middle_frame: bool = True

    # Clip sampling / image preprocessing.
    frames_per_video: int = 16
    resize_short_side: int = 256
    crop_size: int = 224

    # Cache.
    train_cache_views: int = 2
    eval_cache_views: int = 1
    cache_dtype: str = "float16"

    # BiGRU + attention.
    input_dim: int = 512
    gru_hidden: int = 256
    gru_layers: int = 1
    bidirectional: bool = True
    attention_hidden: int = 128
    dropout: float = 0.30

    # Training temporal head.
    batch_size: int = 64
    num_workers: int = 0 if os.name == "nt" else 4
    epochs: int = 35
    lr: float = 1e-3
    weight_decay: float = 1e-4
    label_smoothing: float = 0.10
    grad_clip: float = 1.0
    patience: int = 8

CFG = Config()
seed_everything(CFG.seed)

BASE_DIR = Path("./SemHAR_Gate_Workspace").resolve()
DIRS = {
    "downloads": BASE_DIR / "downloads",
    "data": BASE_DIR / "data",
    "manifests": BASE_DIR / "manifests",
    "cache": BASE_DIR / "cache",
    "checkpoints": BASE_DIR / "checkpoints",
    "results": BASE_DIR / "results",
    "future": BASE_DIR / "future_blocks",
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

DATASET_PATHS = {
    "ucf101": {
        # After automatic extraction the discovery code will overwrite these if needed.
        "video_root": DIRS["data"] / "ucf101",
        "split_root": DIRS["data"] / "ucf101_splits",
    },
    "hmdb51": {
        "video_root": DIRS["data"] / "hmdb51",
        "split_root": DIRS["data"] / "hmdb51_splits",
    },
}

print(json.dumps(asdict(CFG), indent=2, default=str))
print("Workspace:", BASE_DIR)

{
  "seed": 42,
  "fold": 1,
  "quick_mode": false,
  "quick_train_per_class": 8,
  "quick_eval_per_class": 4,
  "auto_download": false,
  "datasets_to_prepare": [
    "ucf101",
    "hmdb51"
  ],
  "datasets_to_train": [
    "ucf101",
    "hmdb51"
  ],
  "val_fraction": 0.1,
  "audit_workers": 1,
  "audit_quick_hash": true,
  "audit_decode_middle_frame": true,
  "frames_per_video": 16,
  "resize_short_side": 256,
  "crop_size": 224,
  "train_cache_views": 2,
  "eval_cache_views": 1,
  "cache_dtype": "float16",
  "input_dim": 512,
  "gru_hidden": 256,
  "gru_layers": 1,
  "bidirectional": true,
  "attention_hidden": 128,
  "dropout": 0.3,
  "batch_size": 64,
  "num_workers": 0,
  "epochs": 35,
  "lr": 0.001,
  "weight_decay": 0.0001,
  "label_smoothing": 0.1,
  "grad_clip": 1.0,
  "patience": 8
}
Workspace: D:\Projects\ML_project\action-recognition-nlp\SemHAR_Gate_Workspace


## 4. Optional dataset download and extraction

### UCF101
The full dataset contains **101 action classes**. The official recognition split files are preserved.

### HMDB51
The full dataset contains **51 action classes**. The original release is distributed as an outer RAR containing one RAR per class, plus official split files.

Automatic HMDB51 extraction requires a RAR-capable executable:
- **Windows:** 7-Zip (`7z.exe`) on `PATH`
- **Linux:** `7z`, `7zz`, or `unrar`
- **macOS:** `brew install p7zip` or an equivalent RAR-capable extractor

If automatic extraction is inconvenient, extract the datasets manually and update `DATASET_PATHS`.

In [5]:
from huggingface_hub import hf_hub_download

HF_SOURCES = {
    "ucf101": {
        "repo": "MichiganNLP/ucf-101",
        "files": ["UCF101.zip", "UCF101TrainTestSplits-RecognitionTask.zip"],
    },
    "hmdb51": {
        "repo": "Serrelab/hmdb51",
        "files": ["hmdb51_org.rar", "test_train_splits.rar"],
    }
}

def download_hf_files(dataset_name: str):
    spec = HF_SOURCES[dataset_name]
    ds_dir = DIRS["downloads"] / dataset_name
    ds_dir.mkdir(parents=True, exist_ok=True)
    outputs = []
    for filename in spec["files"]:
        print(f"Downloading {dataset_name}: {filename}")
        path = hf_hub_download(
            repo_id=spec["repo"],
            filename=filename,
            repo_type="dataset",
            local_dir=str(ds_dir),
        )
        outputs.append(Path(path))
    return outputs

def find_executable(names):
    for name in names:
        p = shutil.which(name)
        if p:
            return p
    return None

def extract_rar(archive: Path, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    exe = find_executable(["7z", "7zz", "unrar"])
    if exe is None:
        raise RuntimeError(
            "No RAR extractor found. Install 7-Zip/unrar and ensure it is on PATH, "
            "or extract HMDB51 manually and set DATASET_PATHS."
        )
    base = Path(exe).name.lower()
    if "unrar" in base:
        cmd = [exe, "x", "-o+", str(archive), str(out_dir) + os.sep]
    else:
        cmd = [exe, "x", "-y", str(archive), f"-o{out_dir}"]
    subprocess.run(cmd, check=True)

def extract_ucf101():
    dl = DIRS["downloads"] / "ucf101"
    video_zip = dl / "UCF101.zip"
    split_zip = dl / "UCF101TrainTestSplits-RecognitionTask.zip"
    video_out = DIRS["data"] / "ucf101_extract"
    split_out = DIRS["data"] / "ucf101_split_extract"

    if not video_out.exists() or not any(video_out.rglob("*.avi")):
        print("Extracting UCF101 videos...")
        video_out.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(video_zip, "r") as zf:
            zf.extractall(video_out)

    if not split_out.exists() or not any(split_out.rglob("trainlist01.txt")):
        print("Extracting UCF101 split files...")
        split_out.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(split_zip, "r") as zf:
            zf.extractall(split_out)

    return video_out, split_out

def extract_hmdb51():
    dl = DIRS["downloads"] / "hmdb51"
    org_rar = dl / "hmdb51_org.rar"
    split_rar = dl / "test_train_splits.rar"

    outer = DIRS["data"] / "hmdb51_outer"
    videos = DIRS["data"] / "hmdb51_videos"
    splits = DIRS["data"] / "hmdb51_split_extract"

    if not outer.exists() or not any(outer.glob("*.rar")):
        print("Extracting HMDB51 outer archive...")
        extract_rar(org_rar, outer)

    videos.mkdir(parents=True, exist_ok=True)
    class_rars = sorted(outer.rglob("*.rar"))
    if not class_rars:
        raise FileNotFoundError("No class RAR files found after extracting hmdb51_org.rar")

    existing_avi = list(videos.rglob("*.avi"))
    if len(existing_avi) < 100:
        print(f"Extracting {len(class_rars)} HMDB51 class archives...")
        for rar_path in tqdm(class_rars):
            class_dir = videos / rar_path.stem
            if not class_dir.exists() or not any(class_dir.glob("*.avi")):
                extract_rar(rar_path, class_dir)

    if not splits.exists() or not any(splits.rglob("*_test_split1.txt")):
        print("Extracting HMDB51 official splits...")
        extract_rar(split_rar, splits)

    return videos, splits

if CFG.auto_download:
    for ds in CFG.datasets_to_prepare:
        download_hf_files(ds)
    extract_ucf101()
    extract_hmdb51()
else:
    print("AUTO_DOWNLOAD=False. Dataset discovery will search configured and extracted locations.")

AUTO_DOWNLOAD=False. Dataset discovery will search configured and extracted locations.


## 5. Robust dataset-root discovery

Dataset archives are not always extracted into identical top-level folder names. Instead of hard-coding one structure, this notebook discovers:
- the directory that contains the largest number of class folders with AVI files;
- the directory containing the official split files.

This makes the notebook easier to reuse across Windows, Linux, Colab, Kaggle, and manually downloaded copies.

In [6]:
def candidate_roots():
    return [
        DIRS["data"],
        DIRS["downloads"],
        BASE_DIR,
    ]

def find_video_root(dataset_name: str) -> Path:
    configured = Path(DATASET_PATHS[dataset_name]["video_root"])
    candidates = []
    if configured.exists():
        candidates.append(configured)
    for root in candidate_roots():
        if root.exists():
            candidates.extend([p for p in root.rglob("*") if p.is_dir()])

    expected = 101 if dataset_name == "ucf101" else 51
    scored = []
    seen = set()

    for p in candidates:
        try:
            rp = str(p.resolve())
        except Exception:
            rp = str(p)
        if rp in seen:
            continue
        seen.add(rp)

        try:
            class_dirs = sum(1 for d in p.iterdir() if d.is_dir() and any(d.glob("*.avi")))
        except Exception:
            continue

        if class_dirs >= 10:
            # Prefer an exact expected class count. This prevents the HMDB51
            # discovery pass from accidentally selecting an already extracted
            # UCF101 directory merely because it contains more class folders.
            score = (abs(class_dirs - expected), -class_dirs)
            scored.append((score, p, class_dirs))

    if not scored:
        raise FileNotFoundError(
            f"Could not discover {dataset_name} video root. "
            f"Edit DATASET_PATHS['{dataset_name}']['video_root']."
        )

    scored.sort(key=lambda x: x[0])
    _, best, best_class_dirs = scored[0]

    if abs(best_class_dirs - expected) > max(3, expected // 10):
        print(
            f"WARNING: {dataset_name} expected about {expected} class folders, "
            f"but discovered {best_class_dirs}. Verify the root before training."
        )

    print(f"{dataset_name}: video root = {best} ({best_class_dirs} class folders detected)")
    return best

def find_split_root(dataset_name: str) -> Path:
    configured = Path(DATASET_PATHS[dataset_name]["split_root"])
    patterns = {
        "ucf101": "trainlist01.txt",
        "hmdb51": "*_test_split1.txt",
    }
    roots = [configured] + candidate_roots()
    matches = []
    for root in roots:
        if root.exists():
            matches.extend(root.rglob(patterns[dataset_name]))
    if not matches:
        raise FileNotFoundError(
            f"Could not discover {dataset_name} split files. "
            f"Edit DATASET_PATHS['{dataset_name}']['split_root']."
        )
    split_root = matches[0].parent
    print(f"{dataset_name}: split root = {split_root}")
    return split_root

DISCOVERED = {}
for ds in CFG.datasets_to_prepare:
    try:
        DISCOVERED[ds] = {
            "video_root": find_video_root(ds),
            "split_root": find_split_root(ds),
        }
    except FileNotFoundError as e:
        print("\n", e)
        print(f"Dataset '{ds}' is not ready. Enable CFG.auto_download or update DATASET_PATHS.\n")


 Could not discover ucf101 video root. Edit DATASET_PATHS['ucf101']['video_root'].
Dataset 'ucf101' is not ready. Enable CFG.auto_download or update DATASET_PATHS.


 Could not discover hmdb51 video root. Edit DATASET_PATHS['hmdb51']['video_root'].
Dataset 'hmdb51' is not ready. Enable CFG.auto_download or update DATASET_PATHS.



## 6. Build official split manifests

A **manifest** is the single source of truth for every later experiment. Each row represents one video and stores:
- dataset;
- path;
- class;
- numeric label;
- official fold;
- train/validation/test partition;
- group/source identifier when available.

### UCF101 validation
UCF101 videos from one group originate from the same long source video. The validation split therefore selects **whole class-specific groups**, not arbitrary clips.

### HMDB51 validation
The official split gives train/test membership but no separate validation set. We carve a deterministic stratified validation subset from the official training set only.

In [7]:
UCF_GROUP_RE = re.compile(r"_g(\d+)_c(\d+)", flags=re.IGNORECASE)

def parse_ucf_group(filename: str, class_name: str) -> str:
    m = UCF_GROUP_RE.search(filename)
    if m:
        return f"{class_name}::g{m.group(1)}"
    return f"{class_name}::{Path(filename).stem}"

def classwise_group_val_split(df_train: pd.DataFrame, val_fraction: float, seed: int):
    rng = random.Random(seed)
    out = df_train.copy()
    out["split"] = "train"
    for class_name, g in out.groupby("class_name"):
        groups = sorted(g["group_id"].unique().tolist())
        rng.shuffle(groups)
        n_val = max(1, int(round(len(groups) * val_fraction)))
        n_val = min(n_val, max(1, len(groups) - 1))
        val_groups = set(groups[:n_val])
        mask = (out["class_name"] == class_name) & out["group_id"].isin(val_groups)
        out.loc[mask, "split"] = "val"
    return out

def build_ucf101_manifest(video_root: Path, split_root: Path, fold: int, val_fraction: float):
    class_file = split_root / "classInd.txt"
    class_lines = [x.strip().split(maxsplit=1) for x in class_file.read_text().splitlines() if x.strip()]
    class_order = [name for _, name in sorted((int(i), n) for i, n in class_lines)]
    class_to_idx = {name: i for i, name in enumerate(class_order)}

    train_file = split_root / f"trainlist{fold:02d}.txt"
    test_file = split_root / f"testlist{fold:02d}.txt"

    rows = []
    for line in train_file.read_text().splitlines():
        if not line.strip():
            continue
        rel, _official_label = line.strip().rsplit(maxsplit=1)
        rel = rel.replace("\\", "/")
        class_name = rel.split("/")[0]
        fp = video_root / Path(rel)
        rows.append({
            "dataset": "ucf101", "filepath": str(fp), "relative_path": rel,
            "video_id": Path(rel).stem, "class_name": class_name,
            "class_id": class_to_idx[class_name], "official_fold": fold,
            "official_split": "train", "split": "train",
            "group_id": parse_ucf_group(Path(rel).name, class_name),
        })

    for line in test_file.read_text().splitlines():
        rel = line.strip().replace("\\", "/")
        if not rel:
            continue
        class_name = rel.split("/")[0]
        fp = video_root / Path(rel)
        rows.append({
            "dataset": "ucf101", "filepath": str(fp), "relative_path": rel,
            "video_id": Path(rel).stem, "class_name": class_name,
            "class_id": class_to_idx[class_name], "official_fold": fold,
            "official_split": "test", "split": "test",
            "group_id": parse_ucf_group(Path(rel).name, class_name),
        })

    df = pd.DataFrame(rows)
    train_part = classwise_group_val_split(
        df[df["official_split"] == "train"].copy(), val_fraction, CFG.seed
    )
    test_part = df[df["official_split"] == "test"].copy()
    return pd.concat([train_part, test_part], ignore_index=True), class_to_idx

def build_hmdb51_manifest(video_root: Path, split_root: Path, fold: int, val_fraction: float):
    split_files = sorted(split_root.glob(f"*_test_split{fold}.txt"))
    if len(split_files) != 51:
        print(f"Warning: expected 51 split files, found {len(split_files)}")

    class_names = sorted([p.name.split(f"_test_split{fold}.txt")[0] for p in split_files])
    class_to_idx = {c: i for i, c in enumerate(class_names)}
    rows = []

    for split_file in split_files:
        class_name = split_file.name.split(f"_test_split{fold}.txt")[0]
        for line in split_file.read_text(errors="ignore").splitlines():
            if not line.strip():
                continue
            filename, status = line.strip().rsplit(maxsplit=1)
            status = int(status)
            if status == 0:
                continue
            official_split = "train" if status == 1 else "test"
            fp = video_root / class_name / filename
            rows.append({
                "dataset": "hmdb51", "filepath": str(fp),
                "relative_path": f"{class_name}/{filename}",
                "video_id": Path(filename).stem, "class_name": class_name,
                "class_id": class_to_idx[class_name], "official_fold": fold,
                "official_split": official_split, "split": official_split,
                "group_id": f"{class_name}::{Path(filename).stem}",
            })

    df = pd.DataFrame(rows)
    train_df = df[df["official_split"] == "train"].copy()
    test_df = df[df["official_split"] == "test"].copy()

    splitter = StratifiedShuffleSplit(
        n_splits=1, test_size=val_fraction, random_state=CFG.seed
    )
    tr_idx, va_idx = next(splitter.split(train_df, train_df["class_id"]))
    train_df.iloc[tr_idx, train_df.columns.get_loc("split")] = "train"
    train_df.iloc[va_idx, train_df.columns.get_loc("split")] = "val"
    test_df["split"] = "test"

    return pd.concat([train_df, test_df], ignore_index=True), class_to_idx

MANIFESTS = {}
CLASS_MAPS = {}

for ds in CFG.datasets_to_prepare:
    if ds not in DISCOVERED:
        continue
    vr = DISCOVERED[ds]["video_root"]
    sr = DISCOVERED[ds]["split_root"]
    if ds == "ucf101":
        df, cmap = build_ucf101_manifest(vr, sr, CFG.fold, CFG.val_fraction)
    else:
        df, cmap = build_hmdb51_manifest(vr, sr, CFG.fold, CFG.val_fraction)

    MANIFESTS[ds] = df
    CLASS_MAPS[ds] = cmap

    df.to_parquet(DIRS["manifests"] / f"{ds}_fold{CFG.fold}_preaudit.parquet", index=False)
    with open(DIRS["manifests"] / f"{ds}_class_to_idx.json", "w") as f:
        json.dump(cmap, f, indent=2)

    print("\n", ds.upper())
    print("Videos:", len(df), "| classes:", df["class_name"].nunique())
    print(pd.crosstab(df["split"], columns="count"))

NameError: name 're' is not defined

## 7. Leakage checks before decoding any video

The first audit is purely structural.

Checks:
1. No identical path may appear in more than one partition.
2. UCF101 source/group IDs must not cross train/validation/test.
3. Every class must occur in every required partition.
4. Manifest class IDs must be stable and one-to-one with class names.

A failed assertion stops the pipeline instead of silently producing optimistic results.

In [ ]:
def structural_leakage_audit(df: pd.DataFrame, dataset_name: str):
    # Path overlap.
    split_paths = {
        s: set(df.loc[df["split"] == s, "filepath"])
        for s in ["train", "val", "test"]
    }
    assert not (split_paths["train"] & split_paths["test"]), "Train/test path overlap!"
    assert not (split_paths["train"] & split_paths["val"]), "Train/val path overlap!"
    assert not (split_paths["val"] & split_paths["test"]), "Val/test path overlap!"

    # UCF group/source overlap is a core protocol requirement.
    if dataset_name == "ucf101":
        split_groups = {
            s: set(df.loc[df["split"] == s, "group_id"])
            for s in ["train", "val", "test"]
        }
        assert not (split_groups["train"] & split_groups["test"]), "UCF train/test group leakage!"
        assert not (split_groups["train"] & split_groups["val"]), "UCF train/val group leakage!"
        assert not (split_groups["val"] & split_groups["test"]), "UCF val/test group leakage!"

    # All classes should be represented.
    n_classes = df["class_name"].nunique()
    for s in ["train", "val", "test"]:
        present = df.loc[df["split"] == s, "class_name"].nunique()
        assert present == n_classes, f"{dataset_name}: only {present}/{n_classes} classes in {s}"

    print(f"{dataset_name}: structural leakage audit PASSED.")

for ds, df in MANIFESTS.items():
    structural_leakage_audit(df, ds)

## 8. Video integrity audit

A path in a split file does not guarantee a usable video. This audit records:
- file existence and size;
- OpenCV readability;
- frame count;
- FPS;
- width/height;
- estimated duration;
- first and optional middle-frame decode success;
- a quick content hash for duplicate detection.

### Quick hash
Instead of hashing every byte of every multi-GB dataset, the audit hashes:
- file size,
- first 1 MiB,
- last 1 MiB.

This is efficient for detecting likely duplicate video files. Suspected duplicates can later be confirmed with a full SHA256 if needed.

In [ ]:
def quick_file_hash(path: Path, chunk: int = 1024 * 1024):
    h = hashlib.sha256()
    size = path.stat().st_size
    h.update(str(size).encode())
    with open(path, "rb") as f:
        first = f.read(chunk)
        h.update(first)
        if size > chunk:
            f.seek(max(0, size - chunk))
            h.update(f.read(chunk))
    return h.hexdigest()

def audit_one_video(filepath: str, middle_decode: bool = True, do_hash: bool = True):
    p = Path(filepath)
    result = {
        "file_exists": p.exists(),
        "file_size_bytes": p.stat().st_size if p.exists() else 0,
        "audit_ok": False,
        "fps": np.nan,
        "frame_count": 0,
        "width": 0,
        "height": 0,
        "duration_sec": np.nan,
        "first_frame_ok": False,
        "middle_frame_ok": False,
        "quick_hash": None,
        "audit_error": "",
    }
    if not p.exists():
        result["audit_error"] = "missing_file"
        return result

    try:
        if do_hash:
            result["quick_hash"] = quick_file_hash(p)

        cap = cv2.VideoCapture(str(p))
        if not cap.isOpened():
            result["audit_error"] = "opencv_open_failed"
            return result

        fps = float(cap.get(cv2.CAP_PROP_FPS))
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        ok_first, _ = cap.read()
        result["first_frame_ok"] = bool(ok_first)

        ok_middle = True
        if middle_decode and n > 1:
            cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, n // 2))
            ok_middle, _ = cap.read()
        result["middle_frame_ok"] = bool(ok_middle)

        cap.release()

        result.update({
            "fps": fps,
            "frame_count": n,
            "width": w,
            "height": h,
            "duration_sec": (n / fps) if fps > 0 else np.nan,
        })
        result["audit_ok"] = (
            n > 0 and w > 0 and h > 0 and result["first_frame_ok"] and result["middle_frame_ok"]
        )
        if not result["audit_ok"]:
            result["audit_error"] = "metadata_or_decode_failed"

    except Exception as e:
        result["audit_error"] = repr(e)

    return result

def audit_manifest(df: pd.DataFrame, dataset_name: str):
    records = []
    for fp in tqdm(df["filepath"].tolist(), desc=f"Audit {dataset_name}"):
        records.append(
            audit_one_video(
                fp,
                middle_decode=CFG.audit_decode_middle_frame,
                do_hash=CFG.audit_quick_hash,
            )
        )
    audit_df = pd.DataFrame(records)
    out = pd.concat([df.reset_index(drop=True), audit_df], axis=1)

    # Suspected content duplicates that cross partitions are especially important.
    if CFG.audit_quick_hash:
        dup = out[out["quick_hash"].notna()].groupby("quick_hash").filter(lambda g: len(g) > 1)
        cross_split_dup = dup.groupby("quick_hash")["split"].nunique()
        bad_hashes = set(cross_split_dup[cross_split_dup > 1].index)
        out["duplicate_cross_split"] = out["quick_hash"].isin(bad_hashes)
    else:
        out["duplicate_cross_split"] = False

    return out

AUDITED = {}
for ds, df in MANIFESTS.items():
    audit_path = DIRS["manifests"] / f"{ds}_fold{CFG.fold}_audited.parquet"
    if audit_path.exists():
        print(f"Loading existing audit: {audit_path}")
        audited = pd.read_parquet(audit_path)
    else:
        audited = audit_manifest(df, ds)
        audited.to_parquet(audit_path, index=False)

    AUDITED[ds] = audited

    print(f"\n{ds.upper()} AUDIT")
    print("Total:", len(audited))
    print("Readable:", int(audited["audit_ok"].sum()))
    print("Failed:", int((~audited["audit_ok"]).sum()))
    print("Cross-split likely duplicates:", int(audited["duplicate_cross_split"].sum()))
    if audited["duplicate_cross_split"].any():
        print("WARNING: investigate cross-split duplicate hashes before research training.")

## 9. Dataset audit summary and EDA

The following summaries become useful later in the paper's dataset section:
- videos per split;
- videos per class;
- duration distribution;
- FPS;
- resolution;
- audit failures.

The model pipeline uses **only videos that pass the decoding audit**. We save the filtered manifest so every later block uses the identical sample set.

In [ ]:
CLEAN = {}

def dataset_summary(df, dataset_name):
    clean = df[df["audit_ok"] & ~df["duplicate_cross_split"]].copy().reset_index(drop=True)
    CLEAN[dataset_name] = clean

    print(f"\n===== {dataset_name.upper()} =====")
    print("Usable videos:", len(clean))
    print("\nSplit counts:")
    print(clean["split"].value_counts().sort_index())
    print("\nClass count:", clean["class_name"].nunique())
    print("\nDuration (sec):")
    print(clean["duration_sec"].describe()[["mean", "std", "min", "50%", "max"]])
    print("\nFPS:")
    print(clean["fps"].describe()[["mean", "std", "min", "50%", "max"]])

    clean_path = DIRS["manifests"] / f"{dataset_name}_fold{CFG.fold}_clean.parquet"
    clean.to_parquet(clean_path, index=False)
    print("Saved:", clean_path)

    # Class balance plot.
    counts = clean.groupby(["class_name", "split"]).size().unstack(fill_value=0)
    fig, ax = plt.subplots(figsize=(14, max(5, 0.18 * len(counts))))
    counts.sort_index().plot(kind="barh", stacked=False, ax=ax)
    ax.set_title(f"{dataset_name.upper()} — videos per class and split")
    ax.set_xlabel("Video count")
    fig.tight_layout()
    plt.show()

    # Duration histogram.
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(clean["duration_sec"].dropna(), bins=40)
    ax.set_title(f"{dataset_name.upper()} — video duration distribution")
    ax.set_xlabel("Duration (seconds)")
    ax.set_ylabel("Videos")
    fig.tight_layout()
    plt.show()

for ds, df in AUDITED.items():
    dataset_summary(df, ds)

## 10. Optional quick mode

`QUICK_MODE=True` keeps a small **stratified subset of every class** for pipeline debugging only. It must never be used for final conference results.

The full clean manifests remain saved and untouched.

In [ ]:
def apply_quick_mode(df: pd.DataFrame):
    if not CFG.quick_mode:
        return df.reset_index(drop=True)

    pieces = []
    rng = np.random.default_rng(CFG.seed)
    for (split, class_id), g in df.groupby(["split", "class_id"]):
        n = CFG.quick_train_per_class if split == "train" else CFG.quick_eval_per_class
        n = min(n, len(g))
        idx = rng.choice(g.index.to_numpy(), size=n, replace=False)
        pieces.append(df.loc[idx])
    return pd.concat(pieces, ignore_index=True)

WORK_MANIFESTS = {ds: apply_quick_mode(df) for ds, df in CLEAN.items()}
for ds, df in WORK_MANIFESTS.items():
    print(ds, df["split"].value_counts().to_dict(), "total =", len(df))

# Part II — Temporal segment sampling and clip preprocessing

## 11. Temporal segment sampling

For a video with \(N\) frames and a desired clip of \(T\) frames, the timeline is divided into \(T\) segments.

- **Training/cache stochastic view:** choose one random frame from each segment.
- **Validation/test deterministic view:** choose the segment center.

This provides broad temporal coverage without decoding every frame.

In [ ]:
def temporal_segment_indices(n_frames: int, num_segments: int, training: bool, rng: random.Random):
    if n_frames <= 1:
        return [0] * num_segments

    edges = np.linspace(0, n_frames, num_segments + 1)
    indices = []
    for i in range(num_segments):
        start = int(math.floor(edges[i]))
        end = int(math.floor(edges[i + 1]))
        start = min(max(start, 0), n_frames - 1)
        end = min(max(end, start + 1), n_frames)
        if training:
            idx = rng.randrange(start, end)
        else:
            idx = min(n_frames - 1, (start + end - 1) // 2)
        indices.append(idx)
    return indices

# Learning check
for n_frames in [7, 16, 100]:
    rng = random.Random(42)
    print(
        f"N={n_frames}",
        "\n train:", temporal_segment_indices(n_frames, CFG.frames_per_video, True, rng),
        "\n eval :", temporal_segment_indices(n_frames, CFG.frames_per_video, False, rng),
        "\n"
    )

## 12. Efficient sampled-frame decoding

OpenCV seeking can occasionally fail on some codecs. The decoder therefore:
1. seeks directly to requested frame indices;
2. if any requested frame fails, falls back to sequentially decoding the full short clip;
3. raises an explicit error if the video still cannot provide usable frames.

Silent replacement with black images is deliberately avoided because it can hide corrupted data.

In [ ]:
def read_sampled_rgb_frames(filepath: str, indices):
    cap = cv2.VideoCapture(filepath)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {filepath}")

    frames = []
    failed = False
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if not ok or frame is None:
            failed = True
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    cap.release()

    if not failed and len(frames) == len(indices):
        return frames

    # Fallback: sequential decode.
    cap = cv2.VideoCapture(filepath)
    all_frames = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        all_frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()

    if not all_frames:
        raise RuntimeError(f"Video produced no frames: {filepath}")

    return [all_frames[min(max(int(i), 0), len(all_frames) - 1)] for i in indices]

## 13. Clip-consistent spatial transforms and normalization

For videos, random crop/flip parameters should be **consistent across all frames of one sampled clip**; otherwise artificial frame-to-frame jitter is introduced.

### Training view
- resize shorter side to 256;
- clip-consistent random crop to 224×224;
- clip-consistent horizontal flip;
- mild clip-consistent brightness/contrast/saturation jitter;
- convert to tensor;
- ImageNet normalization.

### Validation/test view
- resize shorter side to 256;
- center crop 224×224;
- tensor;
- ImageNet normalization.

ImageNet normalization is required because the visual backbone uses ImageNet-pretrained ResNet18.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

class ClipTransform:
    def __init__(self, train: bool, resize_short=256, crop_size=224):
        self.train = train
        self.resize_short = resize_short
        self.crop_size = crop_size

    def _resize(self, img):
        return TF.resize(
            img, self.resize_short,
            interpolation=InterpolationMode.BILINEAR,
            antialias=True
        )

    def __call__(self, frames, rng: random.Random):
        imgs = [self._resize(Image.fromarray(f)) for f in frames]

        if self.train:
            # Clip-consistent random crop after resize.
            w, h = imgs[0].size
            ch = min(self.crop_size, h)
            cw = min(self.crop_size, w)
            top = 0 if h == ch else rng.randint(0, h - ch)
            left = 0 if w == cw else rng.randint(0, w - cw)

            do_flip = rng.random() < 0.5

            # Mild clip-consistent color parameters.
            brightness = rng.uniform(0.90, 1.10)
            contrast   = rng.uniform(0.90, 1.10)
            saturation = rng.uniform(0.90, 1.10)

            out = []
            for img in imgs:
                img = TF.crop(img, top, left, ch, cw)
                if (ch, cw) != (self.crop_size, self.crop_size):
                    img = TF.resize(
                        img, [self.crop_size, self.crop_size],
                        interpolation=InterpolationMode.BILINEAR,
                        antialias=True
                    )
                if do_flip:
                    img = TF.hflip(img)
                img = TF.adjust_brightness(img, brightness)
                img = TF.adjust_contrast(img, contrast)
                img = TF.adjust_saturation(img, saturation)
                x = TF.to_tensor(img)
                x = TF.normalize(x, IMAGENET_MEAN, IMAGENET_STD)
                out.append(x)
        else:
            out = []
            for img in imgs:
                img = TF.center_crop(img, [self.crop_size, self.crop_size])
                x = TF.to_tensor(img)
                x = TF.normalize(x, IMAGENET_MEAN, IMAGENET_STD)
                out.append(x)

        return torch.stack(out, dim=0)  # [T, 3, H, W]

TRAIN_CLIP_TRANSFORM = ClipTransform(True, CFG.resize_short_side, CFG.crop_size)
EVAL_CLIP_TRANSFORM = ClipTransform(False, CFG.resize_short_side, CFG.crop_size)

## 14. Visual sanity check

Before caching thousands of videos, visualize one sampled clip after reversing the ImageNet normalization. This catches:
- BGR/RGB mistakes;
- incorrect crops;
- broken decoding;
- unexpected frame order.

Run this cell for each dataset.

In [ ]:
def denormalize_frame(x):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (x.cpu() * std + mean).clamp(0, 1)

def show_sample_clip(dataset_name: str, split="train", max_frames=8):
    df = WORK_MANIFESTS[dataset_name]
    row = df[df["split"] == split].iloc[0]
    rng = random.Random(CFG.seed)
    idxs = temporal_segment_indices(
        int(row["frame_count"]), CFG.frames_per_video, split == "train", rng
    )
    frames = read_sampled_rgb_frames(row["filepath"], idxs)
    clip = (TRAIN_CLIP_TRANSFORM if split == "train" else EVAL_CLIP_TRANSFORM)(frames, rng)

    n = min(max_frames, clip.shape[0])
    fig, axes = plt.subplots(1, n, figsize=(2 * n, 2.4))
    if n == 1:
        axes = [axes]
    for i in range(n):
        img = denormalize_frame(clip[i]).permute(1, 2, 0).numpy()
        axes[i].imshow(img)
        axes[i].axis("off")
        axes[i].set_title(f"t={i}")
    fig.suptitle(f"{dataset_name.upper()} | {row['class_name']}")
    fig.tight_layout()
    plt.show()

for ds in WORK_MANIFESTS:
    show_sample_clip(ds, "train")

# Part III — Frozen ResNet18 feature caching

## 15. Why cache frame features?

The future SemHAR-Gate blocks repeatedly experiment with:
- temporal models;
- semantic projectors;
- text encoders;
- confidence estimators;
- gating networks.

Re-running ResNet18 for every experiment is wasteful. We therefore cache **frame-level 512-D features**:

\[
[T,3,224,224] \rightarrow ResNet18 \rightarrow [T,512]
\]

The backbone is frozen for Block A. Later, an optional end-to-end fine-tuning phase can be added without changing the manifest or model interfaces.

### Multi-view training cache
Training stores two reproducible stochastic temporal/spatial views per video by default. Validation/test store one deterministic view.

In [ ]:
def build_resnet18_feature_extractor(device):
    weights = ResNet18_Weights.DEFAULT
    model = resnet18(weights=weights)
    model.fc = nn.Identity()
    model.eval().to(device)
    for p in model.parameters():
        p.requires_grad = False
    return model

RESNET18 = build_resnet18_feature_extractor(DEVICE)
print(RESNET18.fc)

## 16. Cache format

Each dataset/split is stored as:
- `features.npy`: `[N, V, T, 512]`, float16 by default;
- `labels.npy`: `[N]`;
- `video_ids.json`;
- `metadata.json`.

`V` is the number of cached views:
- training: 2 by default;
- validation/test: 1.

NumPy memory mapping allows future blocks to read only the batches they need instead of loading the whole cache into RAM.

In [ ]:
def cache_split_features(dataset_name: str, split: str, overwrite=False):
    df = WORK_MANIFESTS[dataset_name]
    split_df = df[df["split"] == split].copy().reset_index(drop=True)

    n = len(split_df)
    views = CFG.train_cache_views if split == "train" else CFG.eval_cache_views
    t = CFG.frames_per_video
    d = CFG.input_dim

    out_dir = DIRS["cache"] / dataset_name / f"fold{CFG.fold}" / split
    out_dir.mkdir(parents=True, exist_ok=True)

    feat_path = out_dir / "features.npy"
    labels_path = out_dir / "labels.npy"
    valid_path = out_dir / "valid.npy"
    ids_path = out_dir / "video_ids.json"
    meta_path = out_dir / "metadata.json"

    if feat_path.exists() and meta_path.exists() and not overwrite:
        print(f"Cache exists: {dataset_name}/{split} -> {feat_path}")
        return out_dir

    dtype = np.float16 if CFG.cache_dtype == "float16" else np.float32
    feats_mm = np.lib.format.open_memmap(
        feat_path, mode="w+", dtype=dtype, shape=(n, views, t, d)
    )
    labels = split_df["class_id"].to_numpy(dtype=np.int64)
    valid = np.zeros(n, dtype=np.bool_)

    for i, row in tqdm(split_df.iterrows(), total=n, desc=f"Cache {dataset_name}/{split}"):
        try:
            for view in range(views):
                # Reproducible but distinct stochastic views.
                local_seed = CFG.seed + i * 1009 + view * 9176 + (0 if split == "train" else 500000)
                rng = random.Random(local_seed)
                idxs = temporal_segment_indices(
                    int(row["frame_count"]), t, split == "train", rng
                )
                frames = read_sampled_rgb_frames(row["filepath"], idxs)
                transform = TRAIN_CLIP_TRANSFORM if split == "train" else EVAL_CLIP_TRANSFORM
                clip = transform(frames, rng).to(DEVICE)

                with torch.inference_mode():
                    with torch.autocast(
                        device_type=DEVICE.type,
                        dtype=torch.float16,
                        enabled=(DEVICE.type == "cuda")
                    ):
                        f = RESNET18(clip)  # [T, 512]
                feats_mm[i, view] = f.detach().float().cpu().numpy().astype(dtype)

            valid[i] = True

        except Exception as e:
            print(f"\nCache failure [{dataset_name}/{split}] {row['filepath']}: {e}")
            feats_mm[i] = 0

    feats_mm.flush()
    np.save(labels_path, labels)
    np.save(valid_path, valid)
    with open(ids_path, "w") as f:
        json.dump(split_df["video_id"].tolist(), f)

    meta = {
        "dataset": dataset_name,
        "fold": CFG.fold,
        "split": split,
        "shape": [n, views, t, d],
        "dtype": CFG.cache_dtype,
        "frames_per_video": t,
        "visual_backbone": "torchvision_resnet18_imagenet1k_v1",
        "image_normalization": {"mean": IMAGENET_MEAN, "std": IMAGENET_STD},
        "cache_train_stochastic": split == "train",
        "valid_count": int(valid.sum()),
        "class_to_idx": CLASS_MAPS[dataset_name],
    }
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    print(f"Saved {dataset_name}/{split}: {int(valid.sum())}/{n} valid")
    return out_dir

CACHE_DIRS = {}
for ds in CFG.datasets_to_prepare:
    if ds not in WORK_MANIFESTS:
        continue
    CACHE_DIRS[ds] = {}
    for split in ["train", "val", "test"]:
        CACHE_DIRS[ds][split] = cache_split_features(ds, split, overwrite=False)

## 17. Cache integrity test

This validates:
- shape;
- finite values;
- label range;
- number of valid videos;
- non-zero feature variance.

A feature cache should never be trusted only because the file exists.

In [ ]:
def inspect_cache(cache_dir: Path):
    with open(cache_dir / "metadata.json") as f:
        meta = json.load(f)
    x = np.load(cache_dir / "features.npy", mmap_mode="r")
    y = np.load(cache_dir / "labels.npy")
    valid = np.load(cache_dir / "valid.npy")

    print(json.dumps({k: meta[k] for k in ["dataset", "split", "shape", "dtype", "valid_count"]}, indent=2))
    print("Feature array:", x.shape, x.dtype)
    print("Labels:", y.shape, y.min(), y.max())
    print("Valid:", valid.sum(), "/", len(valid))

    idx = int(np.flatnonzero(valid)[0])
    sample = np.asarray(x[idx], dtype=np.float32)
    print("Sample finite:", np.isfinite(sample).all())
    print("Sample mean/std:", float(sample.mean()), float(sample.std()))
    assert sample.std() > 0, "Feature cache appears constant."
    assert np.isfinite(sample).all(), "NaN/Inf found in feature cache."

for ds, split_dirs in CACHE_DIRS.items():
    print("\n###", ds.upper())
    for split, p in split_dirs.items():
        inspect_cache(p)

# Part IV — Cached-feature dataset and temporal model

## 18. Cached feature Dataset

During training, each access can select one of the cached stochastic views. Validation and test always use the deterministic view.

Input to the temporal network:

\[
X \in \mathbb{R}^{B\times T\times512}
\]

No image decoding is needed during temporal-model experiments.

In [ ]:
class CachedVideoFeatureDataset(Dataset):
    def __init__(self, cache_dir: Path, train: bool):
        self.cache_dir = Path(cache_dir)
        self.features = np.load(self.cache_dir / "features.npy", mmap_mode="r")
        self.labels = np.load(self.cache_dir / "labels.npy")
        self.valid = np.load(self.cache_dir / "valid.npy")
        with open(self.cache_dir / "video_ids.json") as f:
            self.video_ids = json.load(f)

        self.indices = np.flatnonzero(self.valid)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = int(self.indices[idx])
        n_views = self.features.shape[1]
        view = random.randrange(n_views) if self.train and n_views > 1 else 0

        x = torch.from_numpy(
            np.asarray(self.features[real_idx, view], dtype=np.float32)
        )
        y = int(self.labels[real_idx])
        video_id = self.video_ids[real_idx]
        return x, y, video_id

def build_loaders(dataset_name: str):
    tr = CachedVideoFeatureDataset(CACHE_DIRS[dataset_name]["train"], train=True)
    va = CachedVideoFeatureDataset(CACHE_DIRS[dataset_name]["val"], train=False)
    te = CachedVideoFeatureDataset(CACHE_DIRS[dataset_name]["test"], train=False)

    g = torch.Generator()
    g.manual_seed(CFG.seed)

    train_loader = DataLoader(
        tr, batch_size=CFG.batch_size, shuffle=True,
        num_workers=CFG.num_workers, pin_memory=torch.cuda.is_available(),
        generator=g, drop_last=False
    )
    val_loader = DataLoader(
        va, batch_size=CFG.batch_size, shuffle=False,
        num_workers=CFG.num_workers, pin_memory=torch.cuda.is_available()
    )
    test_loader = DataLoader(
        te, batch_size=CFG.batch_size, shuffle=False,
        num_workers=CFG.num_workers, pin_memory=torch.cuda.is_available()
    )
    return train_loader, val_loader, test_loader

for ds in CACHE_DIRS:
    tr, va, te = build_loaders(ds)
    xb, yb, ids = next(iter(tr))
    print(ds, "batch:", xb.shape, yb.shape, "example ID:", ids[0])

## 19. BiGRU + temporal attention

### BiGRU
The sequence of 512-D frame features is processed in both temporal directions. For offline video classification, both preceding and later sampled frames are available.

### Temporal attention
Not every sampled frame contributes equally. We learn:

\[
e_t = w^\top\tanh(W h_t)
\]

\[
a_t = \operatorname{softmax}(e_t)
\]

\[
h_v = \sum_t a_t h_t
\]

The model returns **three outputs**:
- classification logits;
- pooled video embedding;
- temporal attention weights.

This interface is deliberately future-proof. Later semantic projection and gating blocks will consume `embedding`, `logits`, and `attention`.

In [ ]:
class TemporalAttention(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, h):
        # h: [B, T, D]
        e = self.score(h).squeeze(-1)      # [B, T]
        a = torch.softmax(e, dim=1)        # [B, T]
        pooled = torch.sum(h * a.unsqueeze(-1), dim=1)  # [B, D]
        return pooled, a

class BiGRUAttentionClassifier(nn.Module):
    def __init__(
        self, num_classes: int,
        input_dim=512, hidden_dim=256, num_layers=1,
        bidirectional=True, attention_hidden=128, dropout=0.30
    ):
        super().__init__()

        # Feature normalization across the 512 visual dimensions.
        self.input_norm = nn.LayerNorm(input_dim)

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        temporal_dim = hidden_dim * (2 if bidirectional else 1)
        self.temporal_norm = nn.LayerNorm(temporal_dim)
        self.attention = TemporalAttention(temporal_dim, attention_hidden)

        self.head = nn.Sequential(
            nn.LayerNorm(temporal_dim),
            nn.Dropout(dropout),
            nn.Linear(temporal_dim, num_classes),
        )

        self.embedding_dim = temporal_dim

    def forward(self, x):
        # x: [B, T, 512]
        x = self.input_norm(x)
        h, _ = self.gru(x)                # [B, T, D_temporal]
        h = self.temporal_norm(h)
        embedding, attention = self.attention(h)
        logits = self.head(embedding)
        return {
            "logits": logits,
            "embedding": embedding,
            "attention": attention,
            "frame_states": h,
        }

# Architecture smoke test before real training.
for ds, cmap in CLASS_MAPS.items():
    if ds not in CACHE_DIRS:
        continue
    model = BiGRUAttentionClassifier(
        num_classes=len(cmap),
        input_dim=CFG.input_dim,
        hidden_dim=CFG.gru_hidden,
        num_layers=CFG.gru_layers,
        bidirectional=CFG.bidirectional,
        attention_hidden=CFG.attention_hidden,
        dropout=CFG.dropout,
    ).to(DEVICE)

    dummy = torch.randn(4, CFG.frames_per_video, CFG.input_dim, device=DEVICE)
    out = model(dummy)
    print(
        ds,
        "logits", tuple(out["logits"].shape),
        "embedding", tuple(out["embedding"].shape),
        "attention", tuple(out["attention"].shape),
        "attention sums", out["attention"].sum(dim=1).detach().cpu().numpy()
    )

## 20. Metrics

The checkpoint selection metric is **validation Macro-F1**, not test accuracy.

We report:
- accuracy;
- macro precision;
- macro recall;
- macro F1;
- weighted F1.

Macro-F1 gives each class equal importance and is therefore useful for multi-class HAR.

In [ ]:
def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    mp, mr, mf1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    _, _, wf1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    return {
        "accuracy": float(acc),
        "macro_precision": float(mp),
        "macro_recall": float(mr),
        "macro_f1": float(mf1),
        "weighted_f1": float(wf1),
    }

## 21. Training and validation loop

Block A trains only the temporal classifier over cached visual features.

Research safeguards:
- validation-based checkpointing;
- label-smoothed cross-entropy;
- AdamW;
- gradient clipping;
- cosine learning-rate decay;
- early stopping on validation Macro-F1;
- mixed precision on CUDA;
- history saved to CSV.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)

    all_true, all_pred = [], []
    total_loss, total_n = 0.0, 0

    for x, y, _ids in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=(DEVICE.type == "cuda")
            ):
                out = model(x)
                loss = criterion(out["logits"], y)

            if training:
                if scaler is not None and scaler.is_enabled():
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
                    optimizer.step()

        pred = out["logits"].argmax(dim=1)
        bs = y.size(0)
        total_loss += float(loss.detach().cpu()) * bs
        total_n += bs
        all_true.extend(y.detach().cpu().tolist())
        all_pred.extend(pred.detach().cpu().tolist())

    metrics = compute_metrics(all_true, all_pred)
    metrics["loss"] = total_loss / max(1, total_n)
    return metrics

def train_temporal_model(dataset_name: str):
    train_loader, val_loader, test_loader = build_loaders(dataset_name)
    n_classes = len(CLASS_MAPS[dataset_name])

    model = BiGRUAttentionClassifier(
        num_classes=n_classes,
        input_dim=CFG.input_dim,
        hidden_dim=CFG.gru_hidden,
        num_layers=CFG.gru_layers,
        bidirectional=CFG.bidirectional,
        attention_hidden=CFG.attention_hidden,
        dropout=CFG.dropout,
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(label_smoothing=CFG.label_smoothing)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(1, CFG.epochs)
    )
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

    ckpt_dir = DIRS["checkpoints"] / dataset_name / f"fold{CFG.fold}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = ckpt_dir / "blockA_bigru_attention_best.pt"

    result_dir = DIRS["results"] / dataset_name / f"fold{CFG.fold}" / "blockA"
    result_dir.mkdir(parents=True, exist_ok=True)

    best_f1 = -1.0
    bad_epochs = 0
    history = []

    for epoch in range(1, CFG.epochs + 1):
        t0 = time.time()
        train_m = run_epoch(model, train_loader, criterion, optimizer, scaler)
        val_m = run_epoch(model, val_loader, criterion)
        scheduler.step()

        row = {
            "epoch": epoch,
            "lr": optimizer.param_groups[0]["lr"],
            **{f"train_{k}": v for k, v in train_m.items()},
            **{f"val_{k}": v for k, v in val_m.items()},
            "seconds": time.time() - t0,
        }
        history.append(row)

        print(
            f"[{dataset_name}] epoch {epoch:02d} | "
            f"train loss {train_m['loss']:.4f} F1 {train_m['macro_f1']:.4f} | "
            f"val loss {val_m['loss']:.4f} F1 {val_m['macro_f1']:.4f}"
        )

        if val_m["macro_f1"] > best_f1:
            best_f1 = val_m["macro_f1"]
            bad_epochs = 0
            torch.save({
                "model_state": model.state_dict(),
                "dataset": dataset_name,
                "fold": CFG.fold,
                "config": asdict(CFG),
                "class_to_idx": CLASS_MAPS[dataset_name],
                "embedding_dim": model.embedding_dim,
                "best_val_macro_f1": best_f1,
            }, ckpt_path)
        else:
            bad_epochs += 1

        pd.DataFrame(history).to_csv(result_dir / "train_history.csv", index=False)

        if bad_epochs >= CFG.patience:
            print(f"Early stopping after {epoch} epochs.")
            break

    # Load only the validation-selected checkpoint, then evaluate test once.
    checkpoint = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state"])
    test_m = run_epoch(model, test_loader, criterion)

    with open(result_dir / "test_metrics.json", "w") as f:
        json.dump(test_m, f, indent=2)

    print("\nBest validation Macro-F1:", best_f1)
    print("Test metrics:")
    print(json.dumps(test_m, indent=2))

    return model, (train_loader, val_loader, test_loader), ckpt_path

TRAINED = {}
for ds in CFG.datasets_to_train:
    if ds not in CACHE_DIRS:
        continue
    model, loaders, ckpt = train_temporal_model(ds)
    TRAINED[ds] = {"model": model, "loaders": loaders, "checkpoint": ckpt}

## 22. Test-set predictions and per-class metrics

The notebook now saves a complete prediction table. Later blocks can compare semantic and gated predictions against exactly the same video IDs.

The test set is used **only after** model selection is complete.

In [ ]:
@torch.inference_mode()
def collect_predictions(model, loader):
    model.eval()
    rows = []
    for x, y, ids in tqdm(loader, desc="Predict"):
        x = x.to(DEVICE, non_blocking=True)
        out = model(x)
        probs = torch.softmax(out["logits"], dim=1)
        conf, pred = probs.max(dim=1)

        y_np = y.numpy()
        pred_np = pred.cpu().numpy()
        conf_np = conf.cpu().numpy()

        for vid, yt, yp, cf in zip(ids, y_np, pred_np, conf_np):
            rows.append({
                "video_id": vid,
                "true_class_id": int(yt),
                "pred_class_id": int(yp),
                "visual_max_probability": float(cf),
                "correct": bool(yt == yp),
            })
    return pd.DataFrame(rows)

for ds, obj in TRAINED.items():
    model = obj["model"]
    test_loader = obj["loaders"][2]
    pred_df = collect_predictions(model, test_loader)

    result_dir = DIRS["results"] / ds / f"fold{CFG.fold}" / "blockA"
    pred_df.to_csv(result_dir / "test_predictions.csv", index=False)

    idx_to_class = {v: k for k, v in CLASS_MAPS[ds].items()}
    report = classification_report(
        pred_df["true_class_id"],
        pred_df["pred_class_id"],
        labels=list(range(len(idx_to_class))),
        target_names=[idx_to_class[i] for i in range(len(idx_to_class))],
        output_dict=True,
        zero_division=0,
    )
    pd.DataFrame(report).T.to_csv(result_dir / "per_class_metrics.csv")
    print(ds, "prediction rows:", len(pred_df))

## 23. Confusion matrix

A 101×101 matrix is dense, so the notebook saves both:
- the raw CSV;
- a large normalized image for detailed offline inspection.

Do not infer semantic meaning from the matrix yet; that belongs to later SemHAR-Gate blocks.

In [ ]:
def save_confusion_artifacts(dataset_name: str):
    result_dir = DIRS["results"] / dataset_name / f"fold{CFG.fold}" / "blockA"
    pred_df = pd.read_csv(result_dir / "test_predictions.csv")

    n_classes = len(CLASS_MAPS[dataset_name])
    cm = confusion_matrix(
        pred_df["true_class_id"], pred_df["pred_class_id"],
        labels=list(range(n_classes)), normalize="true"
    )
    np.savetxt(result_dir / "confusion_matrix_normalized.csv", cm, delimiter=",")

    fig_size = 24 if n_classes > 60 else 16
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))
    im = ax.imshow(cm, aspect="auto")
    ax.set_title(f"{dataset_name.upper()} — normalized confusion matrix")
    ax.set_xlabel("Predicted class ID")
    ax.set_ylabel("True class ID")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(result_dir / "confusion_matrix_normalized.png", dpi=180)
    plt.show()

for ds in TRAINED:
    save_confusion_artifacts(ds)

# Part V — Reusable outputs for future SemHAR-Gate blocks

## 24. Export video embeddings, logits, probabilities, attention

This is the most important future-proofing step.

For each train/validation/test video, export:
- `visual_embedding.npy` — pooled BiGRU+attention representation;
- `visual_logits.npy`;
- `visual_probabilities.npy`;
- `temporal_attention.npy`;
- `labels.npy`;
- `video_ids.json`.

Future blocks can implement SBERT/CLIP semantic projection and confidence gating without decoding videos or rerunning ResNet18.

In [ ]:
@torch.inference_mode()
def export_visual_outputs(dataset_name: str, model: nn.Module, loader: DataLoader, split: str):
    model.eval()
    emb_list, logits_list, probs_list, attn_list, labels_list, id_list = [], [], [], [], [], []

    for x, y, ids in tqdm(loader, desc=f"Export {dataset_name}/{split}"):
        x = x.to(DEVICE, non_blocking=True)
        out = model(x)

        logits = out["logits"]
        probs = torch.softmax(logits, dim=1)

        emb_list.append(out["embedding"].float().cpu().numpy())
        logits_list.append(logits.float().cpu().numpy())
        probs_list.append(probs.float().cpu().numpy())
        attn_list.append(out["attention"].float().cpu().numpy())
        labels_list.append(y.numpy())
        id_list.extend(list(ids))

    export_dir = DIRS["future"] / dataset_name / f"fold{CFG.fold}" / split
    export_dir.mkdir(parents=True, exist_ok=True)

    np.save(export_dir / "visual_embedding.npy", np.concatenate(emb_list, axis=0))
    np.save(export_dir / "visual_logits.npy", np.concatenate(logits_list, axis=0))
    np.save(export_dir / "visual_probabilities.npy", np.concatenate(probs_list, axis=0))
    np.save(export_dir / "temporal_attention.npy", np.concatenate(attn_list, axis=0))
    np.save(export_dir / "labels.npy", np.concatenate(labels_list, axis=0))
    with open(export_dir / "video_ids.json", "w") as f:
        json.dump(id_list, f)

    meta = {
        "dataset": dataset_name,
        "fold": CFG.fold,
        "split": split,
        "embedding_dim": int(model.embedding_dim),
        "num_classes": len(CLASS_MAPS[dataset_name]),
        "frames_per_video": CFG.frames_per_video,
        "source": "BlockA_BiGRU_TemporalAttention",
    }
    with open(export_dir / "metadata.json", "w") as f:
        json.dump(meta, f, indent=2)

    print("Saved:", export_dir)

for ds, obj in TRAINED.items():
    model = obj["model"]
    train_loader, val_loader, test_loader = obj["loaders"]
    export_visual_outputs(ds, model, train_loader, "train")
    export_visual_outputs(ds, model, val_loader, "val")
    export_visual_outputs(ds, model, test_loader, "test")

## 25. Temporal-attention learning visualization

This cell helps the student inspect whether the temporal model assigns equal or unequal importance to sampled frames.

At this stage, attention weights are an **internal weighting mechanism**, not a proof of causal explanation.

In [ ]:
@torch.inference_mode()
def show_attention_example(dataset_name: str):
    model = TRAINED[dataset_name]["model"]
    test_loader = TRAINED[dataset_name]["loaders"][2]
    model.eval()

    x, y, ids = next(iter(test_loader))
    x = x.to(DEVICE)
    out = model(x)
    a = out["attention"][0].cpu().numpy()

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.bar(np.arange(len(a)), a)
    ax.set_xlabel("Sampled temporal segment")
    ax.set_ylabel("Attention weight")
    ax.set_title(f"{dataset_name.upper()} temporal attention | video {ids[0]}")
    fig.tight_layout()
    plt.show()

    print("Attention sum:", a.sum())
    print("True class ID:", int(y[0]))
    print("Predicted class ID:", int(out["logits"][0].argmax().cpu()))

for ds in TRAINED:
    show_attention_example(ds)

## 26. Block A completion audit

Do not start Block B until all checks pass.

Block A is complete only if:
- both full dataset manifests are available;
- structural leakage checks pass;
- video audit failures are understood;
- feature caches contain finite, non-constant features;
- BiGRU shapes are correct;
- attention weights sum to approximately 1;
- validation selects the checkpoint;
- test predictions are saved;
- future-block exports exist for train/val/test.

In [ ]:
def block_a_completion_report(dataset_name: str):
    checks = {}

    manifest = DIRS["manifests"] / f"{dataset_name}_fold{CFG.fold}_clean.parquet"
    checks["clean_manifest"] = manifest.exists()

    for split in ["train", "val", "test"]:
        cdir = DIRS["cache"] / dataset_name / f"fold{CFG.fold}" / split
        checks[f"cache_{split}"] = (
            (cdir / "features.npy").exists()
            and (cdir / "metadata.json").exists()
            and (cdir / "valid.npy").exists()
        )

        fdir = DIRS["future"] / dataset_name / f"fold{CFG.fold}" / split
        checks[f"future_export_{split}"] = (
            (fdir / "visual_embedding.npy").exists()
            and (fdir / "visual_logits.npy").exists()
            and (fdir / "visual_probabilities.npy").exists()
            and (fdir / "temporal_attention.npy").exists()
        )

    ckpt = DIRS["checkpoints"] / dataset_name / f"fold{CFG.fold}" / "blockA_bigru_attention_best.pt"
    checks["best_checkpoint"] = ckpt.exists()

    result_dir = DIRS["results"] / dataset_name / f"fold{CFG.fold}" / "blockA"
    checks["test_metrics"] = (result_dir / "test_metrics.json").exists()
    checks["test_predictions"] = (result_dir / "test_predictions.csv").exists()

    print(f"\n===== BLOCK A COMPLETION: {dataset_name.upper()} =====")
    for k, v in checks.items():
        print(f"{'PASS' if v else 'FAIL':4s} | {k}")
    print("Overall:", "PASS" if all(checks.values()) else "NOT COMPLETE")
    return checks

for ds in CFG.datasets_to_train:
    if ds in TRAINED:
        block_a_completion_report(ds)

# 27. How later SemHAR-Gate blocks reuse this notebook

Do **not** change the dataset/split protocol after Block A.

### Block B — Semantic branch
Reuse:
- `*_clean.parquet`
- class mappings
- exported `visual_embedding.npy`
- exported labels/video IDs

Add:
- SBERT/CLIP text encoder;
- prompt bank;
- semantic class prototypes;
- projection MLP;
- cosine/prototype contrastive loss.

### Block C — Confidence and adaptive gate
Reuse:
- `visual_logits.npy`
- `visual_probabilities.npy`
- `temporal_attention.npy`
- semantic logits/probabilities from Block B

Add:
- entropy;
- top-1/top-2 margin;
- JS divergence;
- temporal uncertainty;
- calibration;
- adaptive gate.

### Block D — publication experiments
Reuse the exact same manifests and model interfaces for:
- three seeds;
- fold 1/2/3;
- ablations;
- low-data experiments;
- significance tests;
- rescue/damage analysis.

---

## Student checkpoint questions after completing Block A
The student should be able to answer:
1. Why do UCF101 group IDs matter?
2. Why is validation created only from official training data?
3. Why is one frame sampled from each temporal segment?
4. Why must random spatial transforms be clip-consistent?
5. Why are ImageNet mean/std used with pretrained ResNet18?
6. Why cache frame-level features rather than pooled video features?
7. What does BiGRU add beyond mean pooling?
8. How are temporal attention weights normalized?
9. What is the shape at every stage from `[B,T,3,224,224]` to class logits?
10. Which Block A outputs will be consumed by the semantic branch?

## Dataset references

- **UCF101:** Soomro, Zamir, and Shah, *UCF101: A Dataset of 101 Human Action Classes From Videos in The Wild*.
- **HMDB51:** Kuehne et al., *HMDB: A Large Video Database for Human Motion Recognition*, ICCV 2011.
- Dataset download mirrors used by the optional helper:
  - `MichiganNLP/ucf-101` on Hugging Face.
  - `Serrelab/hmdb51` on Hugging Face.

For final paper experiments, record the exact dataset release, fold, manifest hash, software versions, and random seed.